In [1]:
import os
import re
import subprocess as sp
import tempfile as tem
from pprint import pprint

from ollama import chat


In [2]:
Ollama_MODEL="qwen2.5-coder:7b"

In [3]:
stream=chat(model=Ollama_MODEL,
            messages=[{"role":"user","content":"What is your knowledge cut off and how  good are you with verilog ?"}],
            stream=True)
print("Printing the answer\n")
for chunk in stream:
    print(chunk['message']['content'],end='',flush=True)
print()

Printing the answer

I was trained on data up until October 2023. My expertise is broad and includes many topics, but my proficiency in specific programming languages like Verilog may not be as deep or up-to-date as someone who specializes in it. If you have any questions or need help with Verilog, feel free to ask!


In [4]:
_VERILOG_RULES = """
IMPORTANT VERILOG RULES

Target Language:
- Generate pure Verilog-2001.
- Do NOT generate SystemVerilog.

Syntax Restrictions:

1. Never index a function call result.

WRONG:
result = mul(a,b)[31:0];

CORRECT:
tmp = mul(a,b);
result = tmp[31:0];

2. Never slice expressions.

WRONG:
result = (a+b)[31:0];

CORRECT:
tmp = a+b;
result = tmp;

3. Never declare variables inside procedural blocks.

WRONG:
always @(*) begin
   integer i;
end

CORRECT:
integer i;

always @(*) begin
end

4. Never use:
logic
always_comb
always_ff
always_latch
typedef
enum
struct
union
interface

5. Use only:

reg
wire
assign
always @(*)
always @(posedge clk)
always @(negedge rst)

6. Every case statement must have:

default:

7. Every combinational always block must assign all outputs.

8. Avoid inferred latches.

9. No dynamic arrays.

10. No classes.

11. No assertions.

12. No packages.

13. No generate-if using SystemVerilog syntax.

14. Function outputs must be assigned to temporary variables before slicing.

15. Output only synthesizable Verilog.
16. Never manually sign-extend an arithmetic right shift.

WRONG

result = {32{a[31]}}, a >>> b;

result = {{32{a[31]}},a};

CORRECT

result = $signed(a) >>> b[4:0];

Arithmetic right shift already performs sign extension.
Never concatenate replicated sign bits with >>>.
Never write

unsigned(...)

Always write

$unsigned(...)

-------------------------

Never write

signed(...)

Always write

$signed(...)

-------------------------

Never use

then

Verilog uses

if (...) begin

-------------------------

Never place an if statement outside an always block.

Every procedural statement must be inside

always

initial

task

function

-------------------------

Never use <= inside combinational always blocks.

Use =

-------------------------

Never use VHDL syntax.
"""

In [5]:
def code_gen(prompt:str)->str:
    stream=chat(model=Ollama_MODEL,messages=[{'role':'System',
                                           "content":f"""You are a Senoir Design Engineer and well versed with RTL code, 
                                           provide a clean,synthesizable,well commented,compact Verilog code with respect to the user request.
                                           DO not provide any explanation , provide only the verilog  code.
                                           Follow the rules{_VERILOG_RULES}
                                           """},
                                           {"role":'user',
                                            "content":f"{prompt}"}],
                                            stream=True)
    resp=""
    for chunk in stream:
        content=chunk["message"]["content"]
        print(content,end="",flush=True)
        resp+=content
    return resp

In [20]:
User_inpt=input("Prompt the verilog module  requesting")
response=code_gen(User_inpt)

```verilog
module apb_requester (
    input wire clk,
    input wire rst_n,

    // APB Bus Interface
    output reg [31:0] paddr,
    output reg [31:0] pwdata,
    output reg penable,
    output reg pwrite,
    output reg [2:0] psel,
    input wire pready,
    input wire [31:0] prdata,
    input wire pslverr,

    // User Interface
    input wire [31:0] user_data_in,
    output reg [31:0] user_data_out,
    input wire user_req_valid,
    output reg user_req_ready
);

reg [31:0] addr_reg;
reg [31:0] data_reg;
reg req_valid_reg;

always @(posedge clk or negedge rst_n) begin
    if (!rst_n) begin
        paddr <= 32'b0;
        pwdata <= 32'b0;
        penable <= 1'b0;
        pwrite <= 1'b0;
        psel <= 3'b0;
        user_req_ready <= 1'b0;
    end else begin
        if (user_req_valid && !penable) begin
            addr_reg <= user_data_in[31:0];
            data_reg <= user_data_in[63:32];
            req_valid_reg <= 1'b1;
        end

        if (pready && penable) begin
       

In [21]:
def code_extraction(code:str)->str:
    pattern=r"module\s+\w+.*?endmodule"
    # pattern = r"```(?:verilog|systemverilog)?\s*(module\b.*?endmodule)\s*```"
    matches=re.findall(pattern=pattern,string=code,flags=re.DOTALL)
    clean_code=matches[0]
    return clean_code

In [22]:
clean_Vcode=code_extraction(response)
print(clean_Vcode)

module apb_requester (
    input wire clk,
    input wire rst_n,

    // APB Bus Interface
    output reg [31:0] paddr,
    output reg [31:0] pwdata,
    output reg penable,
    output reg pwrite,
    output reg [2:0] psel,
    input wire pready,
    input wire [31:0] prdata,
    input wire pslverr,

    // User Interface
    input wire [31:0] user_data_in,
    output reg [31:0] user_data_out,
    input wire user_req_valid,
    output reg user_req_ready
);

reg [31:0] addr_reg;
reg [31:0] data_reg;
reg req_valid_reg;

always @(posedge clk or negedge rst_n) begin
    if (!rst_n) begin
        paddr <= 32'b0;
        pwdata <= 32'b0;
        penable <= 1'b0;
        pwrite <= 1'b0;
        psel <= 3'b0;
        user_req_ready <= 1'b0;
    end else begin
        if (user_req_valid && !penable) begin
            addr_reg <= user_data_in[31:0];
            data_reg <= user_data_in[63:32];
            req_valid_reg <= 1'b1;
        end

        if (pready && penable) begin
            if (ps

In [24]:
def linter(code:str)->dict:
    """
    Input code as string and outputs dict consisting returncode , stdout, stderr.
    If return code == 0 then the stderr might be nothing or might have warnings (not errors but warnings). 
    Uses Verilator to run the linter for checking  the syntax of verilog code.
    """
    temp_filename=None
    try:
            with tem.NamedTemporaryFile(
            mode="w",suffix=".v",delete=False,
            ) as temp_file:
                temp_file.write(code)
                temp_file.flush()
    
                temp_filename=temp_file.name
                terminal=[
                          "verilator",
                          "--lint-only",
                          "-Wall",
                          "-Wno-fatal",
                          "-Wno-DECLFILENAME",
                          "-Wno-EOFNEWLINE",
                           temp_filename,
                ]
                result=sp.run(terminal,capture_output=True,text=True,check=False)
                return {
                    "return_code":result.returncode,
                    "stdout":result.stdout,
                    "stderr":result.stderr
                }
    except sp.CalledProcessError as e:
            return {
                "return_code": -1,
                "stdout": "",
                "stderr": f"lint_tool exception: {type(e).__name__}: {e}"
            }
    finally:
            if temp_filename and os.path.exists(temp_filename):
                os.remove(temp_filename)

In [25]:
res=linter(clean_Vcode)
print(res)

{'return_code': 1, 'stdout': '', 'stderr': '%Warning-SELRANGE: /tmp/tmp5fi76lz0.v:37:37: Selection index out of range: 63:32 outside 31:0\n                                           : ... note: In instance \'apb_requester\'\n   37 |             data_reg <= user_data_in[63:32];\n      |                                     ^\n                   ... For warning description see https://verilator.org/warn/SELRANGE?v=5.020\n                   ... Use "/* verilator lint_off SELRANGE */" and lint_on around source to disable this message.\n%Warning-UNUSEDSIGNAL: /tmp/tmp5fi76lz0.v:13:16: Signal is not used: \'pslverr\'\n                                               : ... note: In instance \'apb_requester\'\n   13 |     input wire pslverr,\n      |                ^~~~~~~\n%Warning-UNUSEDSIGNAL: /tmp/tmp5fi76lz0.v:24:5: Signal is not used: \'req_valid_reg\'\n                                              : ... note: In instance \'apb_requester\'\n   24 | reg req_valid_reg;\n      |     ^~~~~~~~~~

In [26]:
def top_module(code:str):
    verilog_file_name="./design.v"
    with open(f"{verilog_file_name}","w") as f:
        f.write(code)
    cmd=["yosys",
                "-p",
                f"""
                read_verilog {verilog_file_name}; 
                hierarchy -auto-top
                """]
    pattern=r"Top module:\s+\\(\w+)"
    res=sp.run(cmd,capture_output=True,text=True,check=False)
    match = re.search(pattern, res.stdout)
    if not match:
        match = re.search(pattern = r"Automatically selected\s+(\w+)\s+as design top module",string=res.stdout)
    if match:
        x= match.group(1)
        return {
        "top_module":x
        }
        


In [27]:
top=top_module(clean_Vcode)

In [28]:
pprint(top)

{'top_module': 'apb_requester'}


In [34]:
def netlist(top:str,module_path:str,lib_path:str)->dict:
    """
    Inupt: 
          1) top-> top moduel name,
          2) module_path -> provide top module path,
          3) lib_path ->  provide path for the tech lib used for netlist generation,
    Output:
        returncode: 0 if successful else any integer when we get error
        stdout: standaroutput for runngin the code in command promt migth be null.
        stderr: error caused due to unsuccessful run.
    """
    try:
        cmd=['yosys',
            '-p',
            f"""
            read_verilog {module_path};
            hierarchy -top {top};
            proc;
            opt;
            memory;
            opt;
            fsm;
            opt;
            techmap;
            opt;
            abc -liberty {lib_path};
            clean;
            write_verilog -noattr {top}_netlist.v;
            show -format dot -prefix {top};
            """
            ]
        out=sp.run(cmd,capture_output=True,text=True,check=False)
    except Exception as e:
        return {
            "netlist_returncode":"-1",
            "netlist_stdout":"Error in netlsit generation.",             
            "netlist_stderr":e}
    return {
        "netlist_returncode":out.returncode,
        "netlist_stdout":out.stdout,
        "netlist_stderr":out.stderr
    }

In [35]:
top_name=top['top_module']
print(top_name)

apb_requester


In [36]:
net_out=netlist(top=top_name,module_path="./design.v",lib_path="../Nangate45/Nangate45_typ.lib")

In [37]:
pprint(net_out)

{'netlist_returncode': 0,
 'netlist_stderr': '',
 'netlist_stdout': '\n'
                   ' '
                   '/----------------------------------------------------------------------------\\\n'
                   ' |  yosys -- Yosys Open SYnthesis '
                   'Suite                                       |\n'
                   ' |  Copyright (C) 2012 - 2026  Claire Xenia Wolf '
                   '<claire@yosyshq.com>         |\n'
                   ' |  Distributed under an ISC-like license, type "license" '
                   'to see terms        |\n'
                   ' '
                   '\\----------------------------------------------------------------------------/\n'
                   ' Yosys 0.62+112 (git sha1 687a36af3, g++ '
                   '13.3.0-6ubuntu2~24.04.1 -fPIC -O3)\n'
                   '\n'
                   '-- Running command `\n'
                   '            read_verilog ./design.v;\n'
                   '            hierarchy -top apb_

In [44]:
def visual_netlist(top:str,path: str):

    svg_path = f"{top}.svg"


    # DOT -> SVG
    out = sp.run(
        ["dot", "-Tsvg", path, "-o", str(svg_path)],
        check=False,
        capture_output=True,
        text=True
    )

    # Check whether Graphviz succeeded
    if out.returncode != 0:
        return {
            "returncode": out.returncode,
            "stdout": out.stdout,
            "stderr": out.stderr
        }

    # Open SVG in default application/browser
    sp.run(
        ["xdg-open", str(svg_path)],
        check=False
    )

    return {
        "returncode": 0,
        "svg_path": str(svg_path)
    }


visual_netlist(top=top_name,path="./apb_requester.dot")

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xe2 in position 113: invalid continuation byte